# Derm8 ISIC Pretraining And PAD Fine-Tuning

Run this notebook in Google Colab with a GPU runtime. It downloads ISIC 2019 from Kaggle, prepares an eight-class dermatology label space, pretrains the image encoder, then fine-tunes the PAD-UFES-20 six-class multimodal model from that checkpoint.

## Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Setup

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess

try:
    from google.colab import userdata
except ImportError:
    userdata = None


def get_config(name, default=None):
    value = os.environ.get(name)
    if value:
        return value
    if userdata is not None:
        try:
            return userdata.get(name) or default
        except Exception:
            return default
    return default


def export_config(name, default=None, required=False):
    value = get_config(name, default)
    if required and not value:
        raise RuntimeError(f'Set {name} in Colab Secrets before training.')
    if value:
        os.environ[name] = str(value)
    return value


REPO_URL = 'https://github.com/SalmaneSossey/mlops-teledermatology.git'
BRANCH = 'main'
REPO_DIR = Path('/content/mlops-teledermatology')
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/mlops-teledermatology')

KAGGLE_ISIC2019_DATASET = 'agsam23/isic-2019-challenge'
ISIC_ROOT = Path('/content/isic_2019')
ISIC_SPLITS_DIR = Path('data/processed/isic_2019_derm8_splits')
DERM8_PRETRAIN_OUTPUT_DIR = DRIVE_PROJECT_DIR / 'runs/isic_2019_derm8_pretrain'

PAD_HF_DATASET_REPO = get_config('PAD_UFES20_HF_REPO_ID', 'SalmaneExploring/pad-ufes-20')
PAD_ROOT = Path('/content/pad_ufes_20')
PAD_IMAGES_DIR = PAD_ROOT / 'all_images'
PAD_SPLITS_DIR = Path('data/processed/splits')
PAD_FINETUNE_OUTPUT_DIR = DRIVE_PROJECT_DIR / 'runs/multimodal_derm8_isic_init'

RUN_DERM8_PRETRAIN = True
RUN_PAD_FINETUNE = True
BUILD_CANDIDATE_BUNDLE = False
CANDIDATE_BUNDLE_DIR = DRIVE_PROJECT_DIR / 'model_bundles/derm8_isic_candidate'

DERM8_EPOCHS = 8
PAD_EPOCHS = 8
BATCH_SIZE = 32
ALLOW_CPU = False

CURRENT_BEST = {
    'test_macro_f1': 0.6902,
    'test_balanced_accuracy': 0.6804,
    'test_high_risk_recall': 0.8902,
    'SCC_recall': 0.2069,
}
TOLERANCE = 0.02

export_config('DAGSHUB_TOKEN', required=True)
export_config('DAGSHUB_USERNAME')
export_config('DAGSHUB_REPO_OWNER', 'SalmaneSossey')
export_config('DAGSHUB_REPO_NAME', 'mlops-teledermatology')
export_config('DAGSHUB_MLFLOW_TRACKING_URI')

DERM8_PRETRAIN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PAD_FINETUNE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CANDIDATE_BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

if REPO_DIR.exists():
    subprocess.run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], cwd='/content', check=True)

os.chdir(REPO_DIR)
print('Working directory:', Path.cwd())
print('ISIC Kaggle dataset:', KAGGLE_ISIC2019_DATASET)
print('Derm8 pretrain output:', DERM8_PRETRAIN_OUTPUT_DIR)
print('PAD fine-tune output:', PAD_FINETUNE_OUTPUT_DIR)


## Install Dependencies

In [ ]:
!pip -q install kaggle mlflow huggingface_hub scikit-learn


## Kaggle, DagsHub, And GPU Checks

In [ ]:
import mlflow
import torch
from mlflow.tracking import MlflowClient

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_json = kaggle_dir / 'kaggle.json'
drive_kaggle_json = Path('/content/drive/MyDrive/kaggle.json')
kaggle_username = get_config('KAGGLE_USERNAME')
kaggle_key = get_config('KAGGLE_KEY')
if kaggle_username and kaggle_key:
    kaggle_json.write_text(json.dumps({'username': kaggle_username, 'key': kaggle_key}))
elif drive_kaggle_json.exists():
    shutil.copyfile(drive_kaggle_json, kaggle_json)
else:
    raise RuntimeError('Set KAGGLE_USERNAME/KAGGLE_KEY in Colab Secrets or place kaggle.json in MyDrive.')
kaggle_json.chmod(0o600)

tracking_uri = os.environ.get('DAGSHUB_MLFLOW_TRACKING_URI') or f"https://dagshub.com/{os.environ['DAGSHUB_REPO_OWNER']}/{os.environ['DAGSHUB_REPO_NAME']}.mlflow"
mlflow.set_tracking_uri(tracking_uri)
experiments = MlflowClient(tracking_uri).search_experiments()
print('MLflow tracking URI:', tracking_uri)
print('Existing experiments:', [(experiment.experiment_id, experiment.name) for experiment in experiments])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
elif (RUN_DERM8_PRETRAIN or RUN_PAD_FINETUNE) and not ALLOW_CPU:
    raise RuntimeError('Select a Colab GPU runtime before starting training.')


## Download ISIC 2019 From Kaggle

In [ ]:
ISIC_ROOT.mkdir(parents=True, exist_ok=True)
!kaggle datasets download -d "{KAGGLE_ISIC2019_DATASET}" -p "{ISIC_ROOT}" --unzip

def find_isic_metadata(root: Path) -> Path:
    candidates = sorted(root.rglob('*GroundTruth*.csv')) + sorted(root.rglob('*groundtruth*.csv'))
    candidates += sorted(root.rglob('*Training*.csv')) + sorted(root.rglob('*.csv'))
    for candidate in candidates:
        text = candidate.read_text(errors='ignore')[:1000]
        if 'MEL' in text and 'BCC' in text and 'SCC' in text:
            return candidate
    raise FileNotFoundError(f'Could not find ISIC 2019 ground-truth CSV under {root}')


def find_image_root(root: Path) -> Path:
    image_dirs = []
    for directory in [root, *[p for p in root.rglob('*') if p.is_dir()]]:
        count = sum(1 for _ in directory.glob('*.jpg')) + sum(1 for _ in directory.glob('*.jpeg')) + sum(1 for _ in directory.glob('*.png'))
        if count:
            image_dirs.append((count, directory))
    if not image_dirs:
        raise FileNotFoundError(f'Could not find ISIC images under {root}')
    return max(image_dirs, key=lambda item: item[0])[1]


ISIC_METADATA_PATH = find_isic_metadata(ISIC_ROOT)
ISIC_IMAGES_DIR = find_image_root(ISIC_ROOT)
print('ISIC metadata:', ISIC_METADATA_PATH)
print('ISIC images dir:', ISIC_IMAGES_DIR)


## Prepare Derm8 Splits

In [ ]:
!python -m src.data.prepare_isic_2019 \
  --metadata-path "{ISIC_METADATA_PATH}" \
  --images-dir "{ISIC_IMAGES_DIR}" \
  --output-dir "{ISIC_SPLITS_DIR}" \
  --label-space derm8


## Train Derm8 Image Encoder

In [ ]:
if RUN_DERM8_PRETRAIN:
    command = [
        'python', '-m', 'src.training.train_image_baseline',
        '--images-dir', str(ISIC_IMAGES_DIR),
        '--splits-dir', str(ISIC_SPLITS_DIR),
        '--output-dir', str(DERM8_PRETRAIN_OUTPUT_DIR),
        '--experiment-name', 'dermatology-8class-isic2019-pretrain',
        '--hf-dataset-repo', KAGGLE_ISIC2019_DATASET,
        '--epochs', str(DERM8_EPOCHS),
        '--batch-size', str(BATCH_SIZE),
        '--sampler', 'weighted_random',
        '--augment-strength', 'current',
    ]
    if ALLOW_CPU:
        command.append('--allow-cpu')
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)
else:
    subprocess.run(['python', '-m', 'src.training.train_image_baseline', '--help'], check=True)


## Download PAD-UFES-20 And Build PAD Splits

In [ ]:
!python -m src.data.download_pad_ufes_20 \
  --repo-id "{PAD_HF_DATASET_REPO}" \
  --output-dir "{PAD_ROOT}" \
  --force

!python -m src.data.make_image_splits \
  --metadata-path "{PAD_ROOT / 'metadata.csv'}" \
  --images-dir "{PAD_IMAGES_DIR}" \
  --output-dir "{PAD_SPLITS_DIR}"


## Fine-Tune PAD Multimodal Model From Derm8 Encoder

In [ ]:
DERM8_CHECKPOINT = DERM8_PRETRAIN_OUTPUT_DIR / 'efficientnet_b0_best.pt'
if not DERM8_CHECKPOINT.exists():
    raise FileNotFoundError(f'Missing derm8 pretrain checkpoint: {DERM8_CHECKPOINT}')

if RUN_PAD_FINETUNE:
    command = [
        'python', '-m', 'src.training.train_multimodal_baseline',
        '--images-dir', str(PAD_IMAGES_DIR),
        '--metadata-path', str(PAD_ROOT / 'metadata.csv'),
        '--splits-dir', str(PAD_SPLITS_DIR),
        '--output-dir', str(PAD_FINETUNE_OUTPUT_DIR),
        '--experiment-name', 'pad-ufes-20-multimodal-derm8-isic-init',
        '--hf-dataset-repo', PAD_HF_DATASET_REPO,
        '--initial-image-checkpoint', str(DERM8_CHECKPOINT),
        '--epochs', str(PAD_EPOCHS),
        '--batch-size', str(BATCH_SIZE),
        '--sampler', 'shuffle',
        '--augment-strength', 'current',
    ]
    if ALLOW_CPU:
        command.append('--allow-cpu')
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)
else:
    subprocess.run(['python', '-m', 'src.training.train_multimodal_baseline', '--help'], check=True)


If Colab runs out of memory, set `BATCH_SIZE = 16` in the setup cell and rerun the training cells.

## Compare PAD Candidate Metrics

In [ ]:
import pandas as pd

metrics_path = PAD_FINETUNE_OUTPUT_DIR / 'multimodal_test_metrics.json'
report_path = PAD_FINETUNE_OUTPUT_DIR / 'multimodal_classification_report.csv'
if not metrics_path.exists():
    raise FileNotFoundError(f'Missing metrics file: {metrics_path}')
if not report_path.exists():
    raise FileNotFoundError(f'Missing classification report: {report_path}')

metrics = json.loads(metrics_path.read_text())
report = pd.read_csv(report_path, index_col=0)
scc_recall = float(report.loc['SCC', 'recall'])
comparison = pd.DataFrame(
    [
        {'metric': 'macro F1', 'current_best': CURRENT_BEST['test_macro_f1'], 'candidate': float(metrics['test_macro_f1']), 'pass_rule': float(metrics['test_macro_f1']) >= CURRENT_BEST['test_macro_f1'] - TOLERANCE},
        {'metric': 'balanced accuracy', 'current_best': CURRENT_BEST['test_balanced_accuracy'], 'candidate': float(metrics['test_balanced_accuracy']), 'pass_rule': float(metrics['test_balanced_accuracy']) >= CURRENT_BEST['test_balanced_accuracy'] - TOLERANCE},
        {'metric': 'high-risk recall', 'current_best': CURRENT_BEST['test_high_risk_recall'], 'candidate': float(metrics['test_high_risk_recall']), 'pass_rule': float(metrics['test_high_risk_recall']) >= CURRENT_BEST['test_high_risk_recall'] - TOLERANCE},
        {'metric': 'SCC recall', 'current_best': CURRENT_BEST['SCC_recall'], 'candidate': scc_recall, 'pass_rule': scc_recall > CURRENT_BEST['SCC_recall']},
    ]
)
display(comparison)
display(report.loc[['BCC', 'MEL', 'SCC'], ['precision', 'recall', 'f1-score', 'support']])
if bool(comparison['pass_rule'].all()):
    print('Candidate passes the derm8 pretraining decision rule.')
else:
    print('Candidate should be reported as an ablation, not promoted as the final model.')


## Optional Candidate Bundle

In [ ]:
if BUILD_CANDIDATE_BUNDLE:
    command = [
        'python', '-m', 'src.inference.build_multimodal_bundle',
        '--metadata-path', str(PAD_ROOT / 'metadata.csv'),
        '--splits-dir', str(PAD_SPLITS_DIR),
        '--output-dir', str(CANDIDATE_BUNDLE_DIR),
        '--checkpoint-path', str(PAD_FINETUNE_OUTPUT_DIR / 'efficientnet_b0_multimodal_best.pt'),
        '--mlflow-run-id', 'derm8-isic-pad-candidate',
    ]
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)
else:
    print('Skipping bundle build. Set BUILD_CANDIDATE_BUNDLE = True after the candidate passes review.')
